# 01. Baseline (EfficientNetB0)
정제 → 증강 → 학습 → 평가. 로직은 `src/`에 있고 이 노트북은 순서대로 호출만 한다.

In [ ]:
# Colab: 저장소를 받고 루트로 이동 (로컬에서는 저장소 루트에서 실행)
# !git clone https://github.com/Junpyodo/brain_tumor_classification.git
# %cd brain_tumor_classification
# !pip install -q -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
from src.config import set_seed, IMG_SIZE
from src.data import load_splits, class_counts, to_tf_dataset
from src.cleaning import find_duplicates, clean_splits
from src.augment import build_augment
from src.model import build_efficientnet, train_two_stage
from src.evaluate import evaluate, plot_history

set_seed()

## 1. 데이터 불러오기와 정제

In [ ]:
splits, label_names = load_splits()
print(class_counts(splits, label_names))
print('정제 전 중복:', {k: v for k, v in find_duplicates(splits).items() if k != 'label_conflicts'})

In [ ]:
splits, report = clean_splits(splits, label_names, size=IMG_SIZE)
print(report)
print(class_counts(splits, label_names))

In [ ]:
train_ds = to_tf_dataset(splits['train'], training=True)
val_ds = to_tf_dataset(splits['validation'])
test_ds = to_tf_dataset(splits['test'])

## 2. 모델과 학습

In [ ]:
augment = build_augment('baseline')
model, base = build_efficientnet(len(label_names), augment)
h1, h2 = train_two_stage(model, base, train_ds, val_ds)
plot_history([h1, h2], 'results/baseline_history.png')

## 3. 최종 평가 (test는 한 번만)

In [ ]:
res = evaluate(model, test_ds, label_names, out_dir='results', tag='baseline')